# Tata Steel AI Hackathon 2026 — Defect Detection in Hot Rolling
## Solution Notebook — Submission v1

**Author:** Ujjawal Shrivastav | **Date:** 2026-05-22

### Approach Summary
Binary classification: predict steel coil defects (4.88% positive rate, 66 defects in 1352 train rows).

Key engineering choices:
- **KNN imputation** for X15 (160 missing values, 7th strongest feature)
- **Ratio features** (X13/X36, X10/X34) encode the physical defect mechanism: rolling force rises, cooling capacity drops
- **Lag features** enabled (CoilID lag-1 autocorrelation = 0.69, sequential production order confirmed)
- **Stage aggregations** across 3 physical process clusters
- **LightGBM** with scale_pos_weight=19.5, 5-fold StratifiedKFold
- **Threshold tuning** on OOF predictions — target: maximize recall subject to precision≥0.90

**V1 honest result:** OOF AUC=0.827, OOF recall=0.924 @ threshold=0.02 (precision=0.078).
The precision target (>90%) is not met at this baseline — v2 will address with SMOTE + anomaly detection ensemble.


## Step 1: Load & Verify Data

In [ ]:
"""
Step 1 — Load & Verify
- Shape / class balance / missing counts
- CoilID sequentiality test (lag feature engineering validity)
- Saves notes_step1.md
"""

import sys
import os
sys.path.insert(0, "/home/ujjwal/Documents/J.A.R.V.I.S./data/hackathons/tata-steel-2026/round_1/ml_harness")

import pandas as pd
import numpy as np
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

from utils.seed import set_seed

set_seed(42)

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR = Path("/home/ujjwal/Documents/J.A.R.V.I.S./data set of tata steel/dataset")
OUT_DIR  = Path("/home/ujjwal/Documents/J.A.R.V.I.S./data/hackathons/tata-steel-2026/round_1/build_v1")
FIG_DIR  = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ── Load ──────────────────────────────────────────────────────────────────────
train  = pd.read_csv(DATA_DIR / "train.csv")
test   = pd.read_csv(DATA_DIR / "test.csv")
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")

print("=" * 60)
print("SHAPES")
print(f"  train  : {train.shape}")
print(f"  test   : {test.shape}")
print(f"  sample : {sample.shape}")
print()

# ── Class balance ─────────────────────────────────────────────────────────────
y = train["Y"]
vc = y.value_counts()
print("CLASS BALANCE (train Y)")
print(f"  Y=0: {vc[0]}  ({vc[0]/len(y)*100:.2f}%)")
print(f"  Y=1: {vc[1]}  ({vc[1]/len(y)*100:.2f}%)")
print(f"  Imbalance ratio: {vc[0]/vc[1]:.1f}:1")
print()

# ── Missing counts ─────────────────────────────────────────────────────────────
missing_train = train.isnull().sum()
missing_test  = test.isnull().sum()
missing_train = missing_train[missing_train > 0]
missing_test  = missing_test[missing_test > 0]

print("MISSING VALUES — TRAIN")
for col, cnt in missing_train.items():
    print(f"  {col}: {cnt} ({cnt/len(train)*100:.2f}%)")
print()

print("MISSING VALUES — TEST")
for col, cnt in missing_test.items():
    print(f"  {col}: {cnt} ({cnt/len(test)*100:.2f}%)")
print()

# ── Sample submission format ──────────────────────────────────────────────────
print("SAMPLE SUBMISSION")
print(f"  columns  : {list(sample.columns)}")
print(f"  rows     : {len(sample)}")
print(f"  test rows: {len(test)}")
print(f"  submission must be: {len(test)} rows × {len(sample.columns)} cols")
print()

# ── CoilID sequentiality test ─────────────────────────────────────────────────
print("=" * 60)
print("COILID SEQUENTIALITY ANALYSIS")
print()

# Sort train by CoilID and examine X13 (top feature)
train_sorted = train.sort_values("CoilID").reset_index(drop=True)

coil_ids = train_sorted["CoilID"].values
x13      = train_sorted["X13"].values

# 1. Check if CoilIDs are monotonically increasing integers or have gaps
coil_diffs = np.diff(coil_ids)
print(f"  CoilID range       : {coil_ids.min()} → {coil_ids.max()}")
print(f"  CoilID diffs — min : {coil_diffs.min():.0f}")
print(f"  CoilID diffs — max : {coil_diffs.max():.0f}")
print(f"  CoilID diffs — mean: {coil_diffs.mean():.2f}")
print(f"  Unique diff values : {len(np.unique(coil_diffs))}")
print()

# 2. Lag-1 autocorrelation of X13
x13_valid = x13[~np.isnan(x13)]
if len(x13_valid) > 2:
    lag1_corr = np.corrcoef(x13_valid[:-1], x13_valid[1:])[0, 1]
    lag2_corr = np.corrcoef(x13_valid[:-2], x13_valid[2:])[0, 1]
    print(f"  X13 lag-1 autocorrelation: {lag1_corr:.4f}")
    print(f"  X13 lag-2 autocorrelation: {lag2_corr:.4f}")

# 3. Spearman of CoilID (sorted) vs X13 — do they covary?
spear_r, spear_p = stats.spearmanr(coil_ids, x13)
print(f"  Spearman(CoilID, X13): r={spear_r:.4f}, p={spear_p:.4f}")

# 4. Check if defect coils cluster by CoilID (are consecutive defects common?)
defect_coil_ids = sorted(train_sorted[train_sorted["Y"] == 1]["CoilID"].values)
if len(defect_coil_ids) > 1:
    defect_gaps = np.diff(defect_coil_ids)
    small_gap = (defect_gaps <= 10).sum()
    print(f"\n  Defect coil IDs — {len(defect_coil_ids)} defects")
    print(f"  Gaps between consecutive defect CoilIDs — min: {defect_gaps.min():.0f}, max: {defect_gaps.max():.0f}, mean: {defect_gaps.mean():.1f}")
    print(f"  Defect pairs with gap ≤ 10 CoilIDs: {small_gap} (out of {len(defect_gaps)})")

# 5. Plot X13 vs CoilID position, coloring defects
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].scatter(
    range(len(train_sorted)),
    train_sorted["X13"],
    c=train_sorted["Y"].map({0: "steelblue", 1: "red"}),
    s=10, alpha=0.6
)
axes[0].set_xlabel("Row index (sorted by CoilID)")
axes[0].set_ylabel("X13 value")
axes[0].set_title("X13 vs CoilID position — Blue=normal, Red=defect")

# Zoom to defect neighborhoods
defect_idx = train_sorted[train_sorted["Y"] == 1].index.tolist()
x10 = train_sorted["X10"].values
axes[1].scatter(
    range(len(train_sorted)),
    train_sorted["X36"],
    c=train_sorted["Y"].map({0: "steelblue", 1: "red"}),
    s=10, alpha=0.6
)
axes[1].set_xlabel("Row index (sorted by CoilID)")
axes[1].set_ylabel("X36 value")
axes[1].set_title("X36 (cooling/tension — drops on defect) vs CoilID position")

plt.tight_layout()
plt.savefig(FIG_DIR / "coilid_sequentiality.png", dpi=120, bbox_inches="tight")
plt.close()
print(f"\n  Figure saved: {FIG_DIR / 'coilid_sequentiality.png'}")

# ── Decision: lag features valid? ─────────────────────────────────────────────
print()
print("=" * 60)
print("VERDICT: IS LAG FEATURE ENGINEERING VALID?")
print()

lag_valid = False

# Criteria: strong lag-1 autocorr OR defect clustering OR Spearman significant
if abs(lag1_corr) > 0.3:
    print(f"  VALID — X13 lag-1 autocorr = {lag1_corr:.4f} > 0.3 threshold")
    lag_valid = True
elif small_gap >= 5:
    print(f"  VALID — {small_gap} defect pairs within 10 CoilIDs = temporal clustering")
    lag_valid = True
elif abs(spear_r) > 0.1 and spear_p < 0.05:
    print(f"  VALID — Spearman(CoilID, X13) = {spear_r:.4f}, p={spear_p:.4f}")
    lag_valid = True
else:
    print(f"  SKIP — No strong evidence of temporal ordering")
    print(f"         lag1_corr={lag1_corr:.4f}, small_gap={small_gap}, Spearman r={spear_r:.4f} p={spear_p:.4f}")

print()

# ── Save notes ────────────────────────────────────────────────────────────────
notes = f"""# Step 1 Notes — Load & Verify

## Data shapes
- train : {train.shape}
- test  : {test.shape}
- sample: {sample.shape}
- Submission must be: {len(test)} rows × {len(sample.columns)} cols

## Class balance
- Y=0: {vc[0]} ({vc[0]/len(y)*100:.2f}%)
- Y=1: {vc[1]} ({vc[1]/len(y)*100:.2f}%)
- Imbalance: {vc[0]/vc[1]:.1f}:1

## Missing values (train)
{chr(10).join(f"- {col}: {cnt} ({cnt/len(train)*100:.2f}%)" for col, cnt in missing_train.items()) if len(missing_train) else "- None"}

## Missing values (test)
{chr(10).join(f"- {col}: {cnt} ({cnt/len(test)*100:.2f}%)" for col, cnt in missing_test.items()) if len(missing_test) else "- None"}

## CoilID sequentiality
- Range: {coil_ids.min()} → {coil_ids.max()}
- Diffs mean: {coil_diffs.mean():.2f}
- X13 lag-1 autocorrelation: {lag1_corr:.4f}
- X13 lag-2 autocorrelation: {lag2_corr:.4f}
- Spearman(CoilID, X13): r={spear_r:.4f}, p={spear_p:.4f}
- Defect clustering (gaps ≤ 10): {small_gap} pairs

## Verdict: Lag features valid?
{'YES — build lag features in Step 2' if lag_valid else 'NO — skip lag features in Step 2 (no reliable sequential ordering)'}

## Key observations
- Top features confirmed: X13, X10, X36, X34 (AUC 0.77–0.83)
- X15 has {missing_train.get("X15", 0)} missing train values ({missing_train.get("X15", 0)/len(train)*100:.2f}%) — use KNN imputation
- Class imbalance is severe — scale_pos_weight = {vc[0]/vc[1]:.1f}
"""

notes_path = OUT_DIR / "notes_step1.md"
notes_path.write_text(notes)
print(f"Notes saved: {notes_path}")

# Save lag verdict for use in step 2
import json
verdict = {"lag_features_valid": lag_valid, "lag1_autocorr": float(lag1_corr), "spearman_r": float(spear_r), "spearman_p": float(spear_p)}
(OUT_DIR / "lag_verdict.json").write_text(json.dumps(verdict, indent=2))
print(f"Lag verdict saved: {OUT_DIR / 'lag_verdict.json'}")
print()
print("Step 1 complete.")


## Step 2: Feature Engineering (49 → 95 features)

In [ ]:
"""
Step 2 — Feature Engineering
Priority order: ratio features → KNN imputation → stage aggregations → row stats → lag features
Saves: train_engineered.parquet, test_engineered.parquet
"""

import sys
sys.path.insert(0, "/home/ujjwal/Documents/J.A.R.V.I.S./data/hackathons/tata-steel-2026/round_1/ml_harness")

import json
import warnings
import pandas as pd
import numpy as np
from scipy.stats import skew
from sklearn.impute import KNNImputer
from pathlib import Path

from utils.seed import set_seed

warnings.filterwarnings("ignore")
set_seed(42)

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR = Path("/home/ujjwal/Documents/J.A.R.V.I.S./data set of tata steel/dataset")
OUT_DIR  = Path("/home/ujjwal/Documents/J.A.R.V.I.S./data/hackathons/tata-steel-2026/round_1/build_v1")

train  = pd.read_csv(DATA_DIR / "train.csv")
test   = pd.read_csv(DATA_DIR / "test.csv")

lag_verdict = json.loads((OUT_DIR / "lag_verdict.json").read_text())
LAG_VALID   = lag_verdict["lag_features_valid"]

print(f"Train: {train.shape}, Test: {test.shape}")
print(f"Lag features valid: {LAG_VALID}")
print()

# Cluster definitions from EDA
CLUSTER3 = ["X4", "X5", "X6", "X7", "X8", "X9", "X10", "X13", "X15", "X29", "X30", "X31", "X32", "X33"]
CLUSTER2 = ["X14", "X19", "X24", "X34", "X35", "X36", "X37", "X38", "X39", "X40", "X41", "X42"]
CLUSTER1 = ["X1", "X2", "X3", "X11", "X12", "X16", "X17", "X18", "X20", "X21", "X22", "X23",
            "X25", "X26", "X27", "X28", "X43", "X44", "X45", "X46", "X47", "X48", "X49"]
ALL_X    = [f"X{i}" for i in range(1, 50)]

# Preserve labels before engineering
y_train = train["Y"].copy()

# Sort both by CoilID (sequential ordering confirmed)
train = train.sort_values("CoilID").reset_index(drop=True)
test  = test.sort_values("CoilID").reset_index(drop=True)
y_train = y_train[train.index].reset_index(drop=True)  # reindex with sorted train
# Actually re-read Y from sorted train
y_train = train["Y"].copy()

def add_engineered_features(df_train, df_test):
    """All engineering done on train-fit parameters, applied to test without leakage."""

    # ── A. KNN IMPUTATION (fit on train, transform both) ─────────────────────
    print("A. KNN imputation for X15 + other missing cols...")
    # Include the KNN neighbors from EDA: X10, X13, X29, X30 + X15 itself
    # Also impute ALL missing cols jointly for consistency
    feat_cols = ALL_X.copy()

    imputer = KNNImputer(n_neighbors=5)
    train_x = df_train[feat_cols].values.copy()
    test_x  = df_test[feat_cols].values.copy()

    train_imputed = imputer.fit_transform(train_x)
    test_imputed  = imputer.transform(test_x)

    df_train = df_train.copy()
    df_test  = df_test.copy()
    df_train[feat_cols] = train_imputed
    df_test[feat_cols]  = test_imputed

    missing_after_train = df_train[feat_cols].isnull().sum().sum()
    missing_after_test  = df_test[feat_cols].isnull().sum().sum()
    print(f"   Missing after imputation — train: {missing_after_train}, test: {missing_after_test}")

    # ── B. RATIO FEATURES (cross-cluster: rolling ÷ cooling) ─────────────────
    print("B. Ratio features (rolling force / cooling divergence)...")
    for df in [df_train, df_test]:
        df["X13_over_X36"]   = df["X13"] / (df["X36"].abs() + 1e-6)
        df["X10_over_X34"]   = df["X10"] / (df["X34"].abs() + 1e-6)
        df["X13_over_X34"]   = df["X13"] / (df["X34"].abs() + 1e-6)   # rising/falling
        df["X30_over_X35"]   = df["X30"] / (df["X35"].abs() + 1e-6)   # force/coiler
        df["X13_minus_X36"]  = df["X13"] - df["X36"]
        df["X10_minus_X34"]  = df["X10"] - df["X34"]
        # Multiplicative interactions (both-rising and both-falling amplify signal)
        df["X13_x_X10"]      = df["X13"] * df["X10"]
        df["X36_x_X34"]      = df["X36"] * df["X34"]
        df["X13_div_X34"]    = df["X13"] / (df["X34"].abs() + 1e-6)
    print(f"   Ratio features added: 9")

    # ── C. STAGE-WISE AGGREGATIONS ────────────────────────────────────────────
    print("C. Stage-wise aggregations (mean, std, min, max per cluster)...")
    for df in [df_train, df_test]:
        for name, cols in [("c3", CLUSTER3), ("c2", CLUSTER2), ("c1", CLUSTER1)]:
            valid_cols = [c for c in cols if c in df.columns]
            sub = df[valid_cols]
            df[f"{name}_mean"] = sub.mean(axis=1)
            df[f"{name}_std"]  = sub.std(axis=1)
            df[f"{name}_min"]  = sub.min(axis=1)
            df[f"{name}_max"]  = sub.max(axis=1)
        # Cross-cluster ratio: rolling cluster mean / cooling cluster mean
        df["c3_over_c2_mean"] = df["c3_mean"] / (df["c2_mean"].abs() + 1e-6)
        df["c3_std_ratio"]    = df["c3_std"] / (df["c3_mean"].abs() + 1e-6)  # coefficient of variation
    print(f"   Cluster aggregation features added: 14")

    # ── D. PER-ROW STATISTICS ─────────────────────────────────────────────────
    print("D. Per-row statistics (mean, std, skew across all X1-X49)...")
    for df in [df_train, df_test]:
        x_vals = df[ALL_X]
        df["row_mean"] = x_vals.mean(axis=1)
        df["row_std"]  = x_vals.std(axis=1)
        df["row_skew"] = x_vals.apply(lambda r: skew(r.dropna()), axis=1)
        df["row_max"]  = x_vals.max(axis=1)
        df["row_min"]  = x_vals.min(axis=1)
        df["row_range"] = df["row_max"] - df["row_min"]
    print(f"   Row stat features added: 6")

    # ── E. WITHIN-CLUSTER Z-SCORES (deviation from cluster mean) ─────────────
    print("E. Within-cluster deviation features for top features...")
    for df in [df_train, df_test]:
        # X13 deviation within Cluster 3
        c3_valid = [c for c in CLUSTER3 if c in df.columns]
        df["X13_c3_zscore"] = (df["X13"] - df["c3_mean"]) / (df["c3_std"] + 1e-6)
        df["X10_c3_zscore"] = (df["X10"] - df["c3_mean"]) / (df["c3_std"] + 1e-6)
        # X36 deviation within Cluster 2
        df["X36_c2_zscore"] = (df["X36"] - df["c2_mean"]) / (df["c2_std"] + 1e-6)
        df["X34_c2_zscore"] = (df["X34"] - df["c2_mean"]) / (df["c2_std"] + 1e-6)
    print(f"   Z-score features added: 4")

    # ── F. LOG TRANSFORM of heavy right-skewed features ───────────────────────
    print("F. Log transforms for heavily skewed features...")
    log_cols = ["X35"]  # EDA: X35 mean~10M for Y=0 vs ~2M for Y=1
    for df in [df_train, df_test]:
        for col in log_cols:
            if col in df.columns:
                df[f"{col}_log"] = np.log1p(df[col].abs())
    print(f"   Log features added: {len(log_cols)}")

    # ── G. LAG FEATURES (conditional on sequentiality verdict) ───────────────
    if LAG_VALID:
        print("G. Lag features (CoilID is sequential — lag-1, lag-2, rolling-5 mean)...")
        lag_features = ["X13", "X10", "X36"]
        for df in [df_train, df_test]:
            for feat in lag_features:
                df[f"{feat}_lag1"]    = df[feat].shift(1).fillna(df[feat].median())
                df[f"{feat}_lag2"]    = df[feat].shift(2).fillna(df[feat].median())
                df[f"{feat}_roll5"]   = df[feat].rolling(5, min_periods=1).mean()
                # Delta: current vs lag-1 (process drift)
                df[f"{feat}_delta1"]  = df[feat] - df[f"{feat}_lag1"]
        print(f"   Lag/rolling features added: {len(lag_features) * 4}")
    else:
        print("G. Lag features SKIPPED (no sequential ordering confirmed)")

    return df_train, df_test


# ── Run engineering ───────────────────────────────────────────────────────────
train_eng, test_eng = add_engineered_features(train, test)

# ── Feature count report ──────────────────────────────────────────────────────
# Drop CoilID and Y from feature list
feature_cols = [c for c in train_eng.columns if c not in ["CoilID", "Y"]]
print()
print(f"Total features after engineering: {len(feature_cols)}")
print(f"  Original: 49")
print(f"  Added   : {len(feature_cols) - 49}")

# ── Sanity check: no NaNs remaining ──────────────────────────────────────────
nan_train = train_eng[feature_cols].isnull().sum().sum()
nan_test  = test_eng[feature_cols].isnull().sum().sum()
print(f"\nNaN check — train: {nan_train}, test: {nan_test}")
if nan_train > 0 or nan_test > 0:
    remaining = train_eng[feature_cols].isnull().sum()
    print("  Remaining NaN cols:", remaining[remaining > 0].to_dict())
    remaining_test = test_eng[feature_cols].isnull().sum()
    print("  Remaining NaN cols (test):", remaining_test[remaining_test > 0].to_dict())

# ── Save ──────────────────────────────────────────────────────────────────────
print()
train_eng.to_parquet(OUT_DIR / "train_engineered.parquet", index=False)
test_eng.to_parquet(OUT_DIR / "test_engineered.parquet", index=False)

print(f"Saved: {OUT_DIR / 'train_engineered.parquet'}  — {train_eng.shape}")
print(f"Saved: {OUT_DIR / 'test_engineered.parquet'}   — {test_eng.shape}")

# Save feature list for downstream scripts
feat_info = {
    "feature_cols": feature_cols,
    "n_features": len(feature_cols),
    "original_x_cols": ALL_X,
}
(OUT_DIR / "feature_list.json").write_text(json.dumps(feat_info, indent=2))
print(f"Saved feature list: {OUT_DIR / 'feature_list.json'}")
print()
print("Step 2 complete.")


## Step 3: LightGBM Baseline — 5-Fold CV

In [ ]:
"""
Step 3 — Baseline LightGBM with 5-fold StratifiedKFold
Outputs: oof_predictions.parquet, models/fold_{i}.lgb, fold metrics
"""

import sys
sys.path.insert(0, "/home/ujjwal/Documents/J.A.R.V.I.S./data/hackathons/tata-steel-2026/round_1/ml_harness")

import json
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from pathlib import Path
from sklearn.metrics import roc_auc_score, precision_recall_curve, recall_score, precision_score

from utils.seed import set_seed
from utils.cv import stratified_kfold

warnings.filterwarnings("ignore")
set_seed(42)

# ── Paths ─────────────────────────────────────────────────────────────────────
OUT_DIR   = Path("/home/ujjwal/Documents/J.A.R.V.I.S./data/hackathons/tata-steel-2026/round_1/build_v1")
MODEL_DIR = OUT_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True)

# ── Load engineered data ──────────────────────────────────────────────────────
train_eng = pd.read_parquet(OUT_DIR / "train_engineered.parquet")
feat_info = json.loads((OUT_DIR / "feature_list.json").read_text())
feature_cols = feat_info["feature_cols"]

X = train_eng[feature_cols].values
y = train_eng["Y"].values
coil_ids = train_eng["CoilID"].values

n_pos = y.sum()
n_neg = (y == 0).sum()
spw   = n_neg / n_pos

print(f"Features: {len(feature_cols)}")
print(f"Train rows: {len(X)}")
print(f"Positives: {n_pos}, Negatives: {n_neg}")
print(f"scale_pos_weight: {spw:.2f}")
print()

# ── LightGBM params ───────────────────────────────────────────────────────────
params = {
    "objective": "binary",
    "metric": "auc",
    "scale_pos_weight": spw,
    "learning_rate": 0.03,
    "num_leaves": 31,
    "min_child_samples": 10,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "reg_alpha": 0.1,
    "reg_lambda": 0.1,
    "random_state": 42,
    "verbose": -1,
    "n_jobs": -1,
}

# ── 5-fold CV ─────────────────────────────────────────────────────────────────
oof_proba = np.zeros(len(X))
fold_metrics = []

print("=" * 60)
print("5-FOLD STRATIFIED CV")
print("=" * 60)

for fold_idx, (tr_idx, val_idx) in enumerate(stratified_kfold(y, n_splits=5, seed=42)):
    X_tr, X_val = X[tr_idx], X[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]

    dtrain = lgb.Dataset(X_tr, label=y_tr)
    dval   = lgb.Dataset(X_val, label=y_val, reference=dtrain)

    callbacks = [
        lgb.early_stopping(stopping_rounds=100, verbose=False),
        lgb.log_evaluation(period=-1),
    ]

    model = lgb.train(
        params,
        dtrain,
        num_boost_round=1000,
        valid_sets=[dval],
        callbacks=callbacks,
    )

    val_proba = model.predict(X_val)
    oof_proba[val_idx] = val_proba

    # Metrics at default threshold 0.5
    val_pred_05 = (val_proba >= 0.5).astype(int)
    auc     = roc_auc_score(y_val, val_proba)
    recall  = recall_score(y_val, val_pred_05, zero_division=0)
    prec    = precision_score(y_val, val_pred_05, zero_division=0)

    # Best threshold on this fold: maximize recall subject to precision >= 0.90
    prec_arr, rec_arr, thr_arr = precision_recall_curve(y_val, val_proba)
    # precision_recall_curve returns precision/recall in decreasing threshold order
    # Find all thresholds where precision >= 0.90
    viable_mask = prec_arr[:-1] >= 0.90
    if viable_mask.any():
        # Among viable thresholds, pick the one with max recall
        best_recall_at_90prec = rec_arr[:-1][viable_mask].max()
        best_thr_at_90prec    = thr_arr[viable_mask][np.argmax(rec_arr[:-1][viable_mask])]
    else:
        best_recall_at_90prec = 0.0
        best_thr_at_90prec    = float("nan")

    # Precision at recall=1.0
    # Find thresholds where recall == 1.0 (catch ALL positives)
    recall1_mask = rec_arr[:-1] >= 1.0
    if recall1_mask.any():
        prec_at_recall1 = prec_arr[:-1][recall1_mask].max()
    else:
        # Lower threshold until we catch all
        prec_at_recall1 = prec_arr[0]  # lowest threshold = all ones

    # Best AUC-based threshold for sanity
    f_scores = 2 * prec_arr[:-1] * rec_arr[:-1] / (prec_arr[:-1] + rec_arr[:-1] + 1e-9)
    best_f1_thr = thr_arr[np.argmax(f_scores)]
    best_f1_pred = (val_proba >= best_f1_thr).astype(int)
    f1_recall = recall_score(y_val, best_f1_pred, zero_division=0)
    f1_prec   = precision_score(y_val, best_f1_pred, zero_division=0)

    print(f"\nFold {fold_idx + 1}:")
    print(f"  Best round     : {model.best_iteration}")
    print(f"  AUC            : {auc:.4f}")
    print(f"  @thr=0.50 — recall={recall:.3f}, precision={prec:.3f}")
    print(f"  @best-F1 thr={best_f1_thr:.3f} — recall={f1_recall:.3f}, precision={f1_prec:.3f}")
    print(f"  Recall @ precision≥0.90: {best_recall_at_90prec:.3f}  (thr={best_thr_at_90prec:.4f})")
    print(f"  Precision @ recall=1.0 : {prec_at_recall1:.4f}")

    fold_metrics.append({
        "fold": fold_idx + 1,
        "auc": float(auc),
        "recall_at_05": float(recall),
        "prec_at_05": float(prec),
        "recall_at_prec90": float(best_recall_at_90prec),
        "thr_at_prec90": float(best_thr_at_90prec) if not np.isnan(best_thr_at_90prec) else None,
        "prec_at_recall1": float(prec_at_recall1),
        "best_round": int(model.best_iteration),
    })

    model.save_model(str(MODEL_DIR / f"fold_{fold_idx}.lgb"))

# ── OOF summary ───────────────────────────────────────────────────────────────
print()
print("=" * 60)
print("OOF SUMMARY")
print("=" * 60)

oof_auc = roc_auc_score(y, oof_proba)

# OOF precision-recall curve
prec_arr_oof, rec_arr_oof, thr_arr_oof = precision_recall_curve(y, oof_proba)

viable_oof = prec_arr_oof[:-1] >= 0.90
if viable_oof.any():
    oof_recall_at_prec90 = rec_arr_oof[:-1][viable_oof].max()
    oof_thr_at_prec90    = thr_arr_oof[viable_oof][np.argmax(rec_arr_oof[:-1][viable_oof])]
else:
    oof_recall_at_prec90 = 0.0
    oof_thr_at_prec90    = float("nan")

# Recall=1.0 on OOF
r1_mask_oof = rec_arr_oof[:-1] >= 1.0
prec_at_r1_oof = prec_arr_oof[:-1][r1_mask_oof].max() if r1_mask_oof.any() else prec_arr_oof[0]

print(f"  OOF AUC                    : {oof_auc:.4f}")
print(f"  OOF Recall @ precision≥0.90: {oof_recall_at_prec90:.4f}  (thr={oof_thr_at_prec90:.4f})")
print(f"  OOF Precision @ recall=1.0 : {prec_at_r1_oof:.4f}")
print()

mean_auc          = np.mean([m["auc"] for m in fold_metrics])
mean_recall_90    = np.mean([m["recall_at_prec90"] for m in fold_metrics])
mean_prec_r1      = np.mean([m["prec_at_recall1"] for m in fold_metrics])

print(f"  Mean fold AUC              : {mean_auc:.4f}")
print(f"  Mean fold recall@prec≥0.90 : {mean_recall_90:.4f}")
print(f"  Mean fold prec@recall=1.0  : {mean_prec_r1:.4f}")

# ── Save OOF predictions ──────────────────────────────────────────────────────
oof_df = pd.DataFrame({
    "CoilID": coil_ids,
    "Y": y,
    "oof_proba": oof_proba,
})
oof_df.to_parquet(OUT_DIR / "oof_predictions.parquet", index=False)
print(f"\nOOF saved: {OUT_DIR / 'oof_predictions.parquet'}")

# Save fold metrics
summary = {
    "fold_metrics": fold_metrics,
    "oof_auc": float(oof_auc),
    "oof_recall_at_prec90": float(oof_recall_at_prec90),
    "oof_thr_at_prec90": float(oof_thr_at_prec90) if not np.isnan(oof_thr_at_prec90) else None,
    "oof_prec_at_recall1": float(prec_at_r1_oof),
    "mean_fold_auc": float(mean_auc),
    "mean_fold_recall_at_prec90": float(mean_recall_90),
    "mean_fold_prec_at_recall1": float(mean_prec_r1),
}
(OUT_DIR / "cv_summary.json").write_text(json.dumps(summary, indent=2))
print(f"CV summary saved: {OUT_DIR / 'cv_summary.json'}")
print()
print("Step 3 complete.")


## Step 4: Threshold Tuning on OOF Predictions

In [ ]:
"""
Step 4 — Threshold tuning on OOF predictions.
The critical step: find the threshold that maximizes recall subject to precision >= 0.90.
Also explores cost-sensitive threshold strategies.
"""

import sys
sys.path.insert(0, "/home/ujjwal/Documents/J.A.R.V.I.S./data/hackathons/tata-steel-2026/round_1/ml_harness")

import json
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import precision_recall_curve, roc_auc_score, recall_score, precision_score

warnings.filterwarnings("ignore")

# ── Paths ─────────────────────────────────────────────────────────────────────
OUT_DIR = Path("/home/ujjwal/Documents/J.A.R.V.I.S./data/hackathons/tata-steel-2026/round_1/build_v1")
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)

# ── Load OOF ──────────────────────────────────────────────────────────────────
oof_df = pd.read_parquet(OUT_DIR / "oof_predictions.parquet")
y_true = oof_df["Y"].values
y_prob = oof_df["oof_proba"].values

n_pos = y_true.sum()
n_neg = (y_true == 0).sum()
print(f"OOF set: {len(y_true)} rows, {n_pos:.0f} positive, {n_neg:.0f} negative")
print(f"OOF AUC: {roc_auc_score(y_true, y_prob):.4f}")
print()

# ── Dense threshold sweep ──────────────────────────────────────────────────────
# Use sklearn's precision_recall_curve which gives exact breakpoints
prec_arr, rec_arr, thr_arr = precision_recall_curve(y_true, y_prob)
# sklearn returns arrays of shape (n_thresholds + 1,) for prec/rec and (n_thresholds,) for thr
# Last element of prec/rec is always precision=1, recall=0 (no-predict edge case — skip)
prec_curve = prec_arr[:-1]
rec_curve  = rec_arr[:-1]
# Add extra sweep for edge cases (very low thresholds)
extra_thr = np.concatenate([np.linspace(0.001, thr_arr[0], 50)[:-1], thr_arr])
extra_prec = []
extra_rec  = []
for t in extra_thr:
    pred = (y_prob >= t).astype(int)
    p = precision_score(y_true, pred, zero_division=0)
    r = recall_score(y_true, pred, zero_division=0)
    extra_prec.append(p)
    extra_rec.append(r)

all_thr  = extra_thr
all_prec = np.array(extra_prec)
all_rec  = np.array(extra_rec)

# ── Key findings ───────────────────────────────────────────────────────────────
print("=" * 60)
print("THRESHOLD ANALYSIS")
print("=" * 60)

# 1. Smallest threshold where precision >= 0.90
prec90_mask = all_prec >= 0.90
if prec90_mask.any():
    # Among thresholds with precision >= 0.90, find the one with MAXIMUM recall
    best_idx   = np.argmax(all_rec[prec90_mask])
    best_thr   = all_thr[prec90_mask][best_idx]
    best_rec   = all_rec[prec90_mask][best_idx]
    best_prec  = all_prec[prec90_mask][best_idx]
    print(f"\n[TARGET]: Max recall at precision>=0.90")
    print(f"  Threshold : {best_thr:.4f}")
    print(f"  Recall    : {best_rec:.4f}  ({best_rec*n_pos:.0f}/{n_pos:.0f} defects caught)")
    print(f"  Precision : {best_prec:.4f}")
    tpr90_thr    = best_thr
    tpr90_recall = best_rec
    tpr90_prec   = best_prec
else:
    print("\n[TARGET]: NO threshold achieves precision>=0.90 on OOF")
    print("  (This is expected given 4.88% positive rate — precision floor is low)")
    tpr90_thr    = None
    tpr90_recall = 0.0
    tpr90_prec   = 0.0

# 2. Largest threshold where recall == 1.0
recall1_mask = all_rec >= 1.0
if recall1_mask.any():
    # Max threshold still giving recall=1.0 → highest precision while catching all
    best_r1_idx  = np.argmax(all_thr[recall1_mask])
    r1_thr       = all_thr[recall1_mask][best_r1_idx]
    r1_prec      = all_prec[recall1_mask][best_r1_idx]
    r1_rec       = all_rec[recall1_mask][best_r1_idx]
    print(f"\n[RECALL=1.0]: Highest threshold where recall=100%")
    print(f"  Threshold : {r1_thr:.4f}")
    print(f"  Precision : {r1_prec:.4f}  ({r1_prec:.1%} precision — needs to be >90%)")
    print(f"  Predicted positives: {(y_prob >= r1_thr).sum() if r1_thr is not None else 'N/A'}")
    print(f"  Gap from target: precision needs {0.90 - r1_prec:.4f} more")
else:
    r1_thr  = None
    r1_prec = None
    print("\n[RECALL=1.0]: Cannot achieve recall=1.0 at any threshold")

# 3. Full sweep report (every 5% threshold)
print("\n\nFULL THRESHOLD SWEEP (representative points):")
print(f"{'Threshold':>10} | {'Recall':>8} | {'Precision':>10} | {'TP':>4} | {'FP':>4} | {'FN':>4} | {'Passes?':>8}")
print("-" * 65)
for thr in [0.01, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.60, 0.70, 0.80, 0.90]:
    pred = (y_prob >= thr).astype(int)
    tp   = ((pred == 1) & (y_true == 1)).sum()
    fp   = ((pred == 1) & (y_true == 0)).sum()
    fn   = ((pred == 0) & (y_true == 1)).sum()
    p    = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    passes = "YES" if (r >= 1.0 and p >= 0.90) else ("RECALL" if r >= 1.0 else ("PREC" if p >= 0.90 else "MISS"))
    print(f"{thr:>10.2f} | {r:>8.3f} | {p:>10.3f} | {tp:>4} | {fp:>4} | {fn:>4} | {passes:>8}")

# ── Decision logic ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("PRODUCTION THRESHOLD DECISION")
print("=" * 60)

# Strategy: primary goal is recall, precision is constraint
# Given the hard imbalance, we'll use the threshold that gives highest recall at prec>=0.90
# If no threshold achieves prec>=0.90 AND recall=1.0, we use the recall-maximizing threshold

if tpr90_thr is not None and tpr90_recall >= 1.0:
    chosen_thr = tpr90_thr
    rationale  = f"Both criteria met: recall=1.0, precision={tpr90_prec:.3f}>=0.90"
    criteria_met = True
elif tpr90_thr is not None:
    chosen_thr = tpr90_thr
    rationale  = f"Max recall at prec>=0.90 constraint: recall={tpr90_recall:.3f}, precision={tpr90_prec:.3f}. Gap: need {1.0-tpr90_recall:.3f} more recall."
    criteria_met = False
elif r1_thr is not None:
    # The ultra-low threshold catches all defects but precision is near base rate.
    # This causes test set collapse (all rows flagged). Use thr=0.02 instead:
    # OOF recall=0.924 @ precision=0.078 — best recall achievable without flagging everything on test.
    chosen_thr   = 0.02
    _pred_02     = (y_prob >= 0.02).astype(int)
    _rec_02      = recall_score(y_true, _pred_02, zero_division=0)
    _prec_02     = precision_score(y_true, _pred_02, zero_division=0)
    rationale    = (
        f"Recall=1.0 at thr=0.0028 BUT test proba floor=0.0168 causes all-positive collapse. "
        f"Using thr=0.02 instead: OOF recall={_rec_02:.3f}, precision={_prec_02:.3f}. "
        f"Test positives ~17%. Prioritize recall, accept precision gap — v2 will close it."
    )
    criteria_met = False
else:
    # Use very low threshold to maximize recall
    chosen_thr = all_thr[np.argmax(all_rec)]
    rationale  = "No clean solution. Using minimum threshold to maximize recall."
    criteria_met = False

chosen_pred  = (y_prob >= chosen_thr).astype(int)
chosen_rec   = recall_score(y_true, chosen_pred, zero_division=0)
chosen_prec  = precision_score(y_true, chosen_pred, zero_division=0)
chosen_tp    = ((chosen_pred == 1) & (y_true == 1)).sum()
chosen_fp    = ((chosen_pred == 1) & (y_true == 0)).sum()
chosen_fn    = ((chosen_pred == 0) & (y_true == 1)).sum()

print(f"\n  Chosen threshold : {chosen_thr:.4f}")
print(f"  Recall           : {chosen_rec:.4f}  (TP={chosen_tp}, FN={chosen_fn})")
print(f"  Precision        : {chosen_prec:.4f}  (FP={chosen_fp})")
print(f"  Criteria met     : {criteria_met}")
print(f"  Rationale        : {rationale}")

# ── Plot ───────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Precision-Recall curve
axes[0].plot(rec_curve, prec_curve, "b-", lw=2, label="Precision-Recall curve")
axes[0].axhline(0.90, color="red", linestyle="--", lw=1.5, label="Precision=0.90 target")
axes[0].axvline(1.0,  color="green", linestyle="--", lw=1.5, label="Recall=1.0 target")
axes[0].scatter([chosen_rec], [chosen_prec], c="orange", s=150, zorder=5, label=f"Chosen (thr={chosen_thr:.3f})")
if r1_thr is not None:
    axes[0].scatter([r1_rec], [r1_prec], c="purple", s=150, zorder=5, marker="^", label=f"Recall=1.0 point (prec={r1_prec:.3f})")
axes[0].set_xlabel("Recall")
axes[0].set_ylabel("Precision")
axes[0].set_title("OOF Precision-Recall Curve")
axes[0].legend(fontsize=9)
axes[0].set_xlim([0, 1.05])
axes[0].set_ylim([0, 1.05])
axes[0].grid(True, alpha=0.3)

# Right: Recall and Precision vs Threshold
axes[1].plot(all_thr, all_rec,  "b-",  lw=2, label="Recall")
axes[1].plot(all_thr, all_prec, "r-",  lw=2, label="Precision")
axes[1].axhline(0.90, color="red",   linestyle="--", lw=1, alpha=0.7, label="Precision=0.90 target")
axes[1].axhline(1.00, color="blue",  linestyle="--", lw=1, alpha=0.7, label="Recall=1.0 target")
axes[1].axvline(chosen_thr, color="orange", linestyle="--", lw=2, label=f"Chosen thr={chosen_thr:.3f}")
axes[1].set_xlabel("Threshold")
axes[1].set_ylabel("Score")
axes[1].set_title("Recall & Precision vs Threshold")
axes[1].legend(fontsize=9)
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1.05])
axes[1].grid(True, alpha=0.3)

plt.suptitle(f"OOF Threshold Analysis — AUC={roc_auc_score(y_true, y_prob):.4f}", fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / "recall_precision_curve.png", dpi=120, bbox_inches="tight")
plt.close()
print(f"\nFigure saved: {FIG_DIR / 'recall_precision_curve.png'}")

# ── Save chosen threshold ──────────────────────────────────────────────────────
threshold_doc = {
    "chosen_threshold": float(chosen_thr),
    "chosen_recall": float(chosen_rec),
    "chosen_precision": float(chosen_prec),
    "tp": int(chosen_tp),
    "fp": int(chosen_fp),
    "fn": int(chosen_fn),
    "criteria_met": criteria_met,
    "rationale": rationale,
    "target_recall": 1.0,
    "target_precision": 0.90,
    "oof_auc": float(roc_auc_score(y_true, y_prob)),
    "recall_at_r1_0": {
        "threshold": float(r1_thr) if r1_thr else None,
        "precision": float(r1_prec) if r1_prec else None,
    },
    "max_recall_at_prec90": {
        "threshold": float(tpr90_thr) if tpr90_thr else None,
        "recall": float(tpr90_recall),
        "precision": float(tpr90_prec),
    },
}

(OUT_DIR / "chosen_threshold.json").write_text(json.dumps(threshold_doc, indent=2))
print(f"Threshold saved: {OUT_DIR / 'chosen_threshold.json'}")
print()
print("Step 4 complete.")


## Step 5: Final Predictions & Submission

In [ ]:
"""
Step 5 — Final predictions on test set.
- Average 5 fold models
- Apply chosen threshold
- Generate submission CSV (339 rows × 2 cols)
- Save raw probabilities for v2 iteration
"""

import sys
sys.path.insert(0, "/home/ujjwal/Documents/J.A.R.V.I.S./data/hackathons/tata-steel-2026/round_1/ml_harness")

import json
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from pathlib import Path

from utils.submission import write_submission

warnings.filterwarnings("ignore")

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR  = Path("/home/ujjwal/Documents/J.A.R.V.I.S./data set of tata steel/dataset")
OUT_DIR   = Path("/home/ujjwal/Documents/J.A.R.V.I.S./data/hackathons/tata-steel-2026/round_1/build_v1")
MODEL_DIR = OUT_DIR / "models"

# ── Load ──────────────────────────────────────────────────────────────────────
test_eng  = pd.read_parquet(OUT_DIR / "test_engineered.parquet")
feat_info = json.loads((OUT_DIR / "feature_list.json").read_text())
thr_doc   = json.loads((OUT_DIR / "chosen_threshold.json").read_text())
sample    = pd.read_csv(DATA_DIR / "sample_submission.csv")
test_orig = pd.read_csv(DATA_DIR / "test.csv")

feature_cols = feat_info["feature_cols"]
threshold    = thr_doc["chosen_threshold"]

X_test = test_eng[feature_cols].values
coil_ids_test = test_eng["CoilID"].values

print(f"Test rows   : {len(X_test)}")
print(f"Features    : {len(feature_cols)}")
print(f"Threshold   : {threshold:.4f}")
print(f"Criteria met: {thr_doc['criteria_met']}")
print()

# ── Load 5 fold models and predict ────────────────────────────────────────────
fold_probas = []
for i in range(5):
    model_path = MODEL_DIR / f"fold_{i}.lgb"
    model = lgb.Booster(model_file=str(model_path))
    proba = model.predict(X_test)
    fold_probas.append(proba)
    print(f"  Fold {i+1} — predicted {(proba >= threshold).sum()} positives (mean proba={proba.mean():.4f})")

# Average ensemble
test_proba_mean = np.mean(fold_probas, axis=0)
test_proba_std  = np.std(fold_probas, axis=0)

print(f"\nEnsemble mean proba — min={test_proba_mean.min():.4f}, max={test_proba_mean.max():.4f}, mean={test_proba_mean.mean():.4f}")
print(f"Ensemble std         — mean={test_proba_std.mean():.4f} (model agreement)")

# ── Apply threshold ────────────────────────────────────────────────────────────
test_pred = (test_proba_mean >= threshold).astype(int)

n_pred_pos = test_pred.sum()
n_pred_neg = (test_pred == 0).sum()
pred_rate  = n_pred_pos / len(test_pred)

print(f"\nPredictions:")
print(f"  Y=0: {n_pred_neg}")
print(f"  Y=1: {n_pred_pos}")
print(f"  Predicted positive rate: {pred_rate:.2%}")
print(f"  (Train positive rate was: 4.88%)")

# ── Validate CoilID alignment ──────────────────────────────────────────────────
# test_eng was sorted by CoilID in Step 2 — confirm match with test.csv sorted
test_orig_sorted = test_orig.sort_values("CoilID")
assert list(test_eng["CoilID"].values) == list(test_orig_sorted["CoilID"].values), \
    "CoilID mismatch between engineered test and original test!"
print("\nCoilID alignment: PASSED")

# ── Build submission dataframe ────────────────────────────────────────────────
# Must match sample_submission.csv format: CoilID, Y
# But we need the original CoilID order from test.csv (not sorted)
# Actually — let's align to the ORIGINAL test.csv order
test_orig_coil_order = pd.read_csv(DATA_DIR / "test.csv")["CoilID"].values

# Build coilid → pred map
coilid_to_pred  = dict(zip(coil_ids_test, test_pred))
coilid_to_proba = dict(zip(coil_ids_test, test_proba_mean))

# Apply in original test.csv order
y_submission = [coilid_to_pred[c] for c in test_orig_coil_order]
p_submission = [coilid_to_proba[c] for c in test_orig_coil_order]

submission_df = pd.DataFrame({
    "CoilID": test_orig_coil_order,
    "Y": y_submission,
})

print(f"\nSubmission shape: {submission_df.shape}")
print(f"Y=1 count: {submission_df['Y'].sum()}")
print(f"Y=0 count: {(submission_df['Y'] == 0).sum()}")
print(f"Nulls: {submission_df.isnull().sum().sum()}")

# ── Validate row count (can't use write_submission validator since sample has only 10 rows) ──
assert len(submission_df) == len(test_orig_coil_order), f"Row count mismatch: {len(submission_df)} vs {len(test_orig_coil_order)}"
assert list(submission_df.columns) == ["CoilID", "Y"], f"Column mismatch: {list(submission_df.columns)}"
assert submission_df["Y"].isnull().sum() == 0, "Nulls in Y column!"

# Check CoilIDs match
expected_coils = set(test_orig_coil_order)
actual_coils   = set(submission_df["CoilID"])
assert expected_coils == actual_coils, f"CoilID set mismatch!"
print("Validation: PASSED (339 rows, 2 cols, no nulls, all CoilIDs present)")

# ── Save ──────────────────────────────────────────────────────────────────────
submission_path = OUT_DIR / "expected_submission.csv"
submission_df.to_csv(submission_path, index=False)
print(f"\nSubmission saved: {submission_path}")

# Raw probabilities for v2
proba_df = pd.DataFrame({
    "CoilID": test_orig_coil_order,
    "proba": p_submission,
    "Y_pred": y_submission,
})
proba_df.to_parquet(OUT_DIR / "test_probas.parquet", index=False)
print(f"Raw probas saved: {OUT_DIR / 'test_probas.parquet'}")

# Quick preview
print("\n--- Submission preview (first 10 rows) ---")
print(submission_df.head(10).to_string(index=False))
print(f"\n--- Rows predicted as DEFECT (Y=1): {submission_df[submission_df['Y']==1]['CoilID'].tolist()[:20]} ---")
print()
print("Step 5 complete.")


## Results Summary

| Metric | V1 Value |
|--------|----------|
| Train positives | 66 / 1352 (4.88%) |
| Features engineered | 95 (from 49 original) |
| OOF AUC | 0.827 |
| OOF Recall @ thr=0.02 | 0.924 |
| OOF Precision @ thr=0.02 | 0.078 |
| Test predicted defects | 157 / 339 |
| Full criteria met? | No — v2 needed |

### Top 3 fixes for v2
1. **SMOTE oversampling** — synthetic minority examples to sharpen boundary
2. **Anomaly detection ensemble** — Isolation Forest as meta-feature
3. **LightGBM stability** — fix dormant fold issue with `is_unbalance=True` and `min_data_in_leaf=5`
